In [5]:
import os
os.environ["PYTHONUTF8"] = "1"

In [6]:
import sys
import types

# Create a fake module to satisfy ragas's broken hardcoded import
fake_module = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:
    def __init__(self, *args, **kwargs):
        raise ImportError("ChatVertexAI is not actually available — install langchain-google-vertexai if you need it.")

fake_module.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = fake_module

In [7]:
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop import MultiHopSpecificQuerySynthesizer

In [8]:
import os
import logging

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

SCHEMES_PDF = "data/sampled_schemes.pdf"

llm = ChatOpenAI(model="gpt-5.6-luna")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# gpt-5.6-luna (like other reasoning-tier models) only accepts the default temperature=1,
# but Ragas' testset pipeline samples generations at non-default temperatures internally,
# so a temperature-flexible model is used just for testset generation.
ragas_llm = ChatOpenAI(model="gpt-5.4-nano")



In [9]:
os.environ["TORCHDYNAMO_DISABLE"] = "1"
logging.getLogger("docling").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(do_ocr=False)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

schemes_doc = converter.convert(SCHEMES_PDF).document
schemes_md = schemes_doc.export_to_markdown()

print(f"{len(schemes_md):,} chars")
print(schemes_md[:500])

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 6469.32it/s]


16,488 chars
## CENTRALLY SPONSORED SCHEMES

## NATIONAL FOOD SECURITY MISSION

The scheme is being implemented from 2007 ­ 08 as a Central Sector Scheme with an objective to enhance production &amp; productivity of Rice &amp; Pulse crops in selected districts of the state in Mission Mode. Besides, this embodies Sub ­ Schemes for enhancing production &amp; productivity of Coarse Cereals under NFSM ­ Coarse Cereals, and Cotton, Jute and Sugarcane under NFSM ­ Commercial crops, NFSM ­ Nutricereals and NFSM ­ O


In [10]:
from langchain_text_splitters import MarkdownTextSplitter

splitter = MarkdownTextSplitter(chunk_size=600, chunk_overlap=120)
chunks = splitter.create_documents([schemes_md], metadatas=[{"source": SCHEMES_PDF}])

In [11]:
len(chunks)

39

In [12]:

from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop import MultiHopSpecificQuerySynthesizer

generator = TestsetGenerator.from_langchain(llm=ragas_llm, embedding_model=embeddings)

# MultiHopAbstractQuerySynthesizer (part of Ragas' default distribution) asks the LLM for a
# nested list-of-lists ("concept combinations"); gpt-5.4-nano sometimes flattens this to a
# plain list of strings, which fails Pydantic validation. Dropping it and keeping only the
# single-hop and multi-hop-specific synthesizers (which use simpler output schemas) avoids
# that failure mode while still producing a mix of single-hop and multi-hop questions.
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 0.6),
    (MultiHopSpecificQuerySynthesizer(llm=ragas_llm), 0.4),
]

In [14]:
testset = generator.generate_with_langchain_docs(
    chunks, testset_size=10, query_distribution=query_distribution
)

eval_df = testset.to_pandas()
print(eval_df.columns.tolist())
eval_df.head()

Applying CustomNodeFilter:   0%|          | 0/39 [00:00<?, ?it/s]Node 14870911-009c-44aa-8a5f-7a4602064341 does not have a summary. Skipping filtering.
Node 071620c2-26ef-4dff-a0fc-1741b8e1ecab does not have a summary. Skipping filtering.
Node 5c3bff8f-bfb3-4385-bf67-8fc4d1fce570 does not have a summary. Skipping filtering.
Node 52747c26-7dbb-4b3e-a74d-4066d22a6ea6 does not have a summary. Skipping filtering.
Node 265be0df-af8a-4982-8430-c0477a5d29f0 does not have a summary. Skipping filtering.
Node 5b40cb45-bd55-4fc0-8ad9-81ccf5d0a659 does not have a summary. Skipping filtering.
Node cc2f4813-b54f-48d4-bd7f-9fc299fc63d8 does not have a summary. Skipping filtering.
Node 01386545-6f98-46f8-82e4-e0759d15adf0 does not have a summary. Skipping filtering.
Node 042ee6ee-0e0b-43c4-9e1c-9e275df003e4 does not have a summary. Skipping filtering.
Node 087937f7-2a82-456c-844b-60ac8b59bb22 does not have a summary. Skipping filtering.
Node 9bdc9cd2-9d44-460a-9dfc-7eb5eb131056 does not have a summary

['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is NATIONAL FOOD SECURITY MISSION?,[## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL...,The context only mentions NATIONAL FOOD SECURI...,María Ortega,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,"How was the scheme implemented from 2007-08, a...",[The scheme is being implemented from 2007 ­ 0...,The scheme is being implemented from 2007-08 a...,Dr. Kavya Menon,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,Prices ka kya matlab hai aur ye kaise under ch...,[Objectives: Increase food security by steppin...,"Context ke mutabiq, objectives me ""prices rema...",Dr. Kavya Menon,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,Farm machineries distribution under NMSA and w...,[-  Cluster Demonstrations - test beds for de...,Farm Machineries refers to the distribution of...,Dr. Kavya Menon,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
4,What is the purpose of setting up project mana...,[-  Plant Protection - Subsidised distributio...,The context states that part of the program in...,María Ortega,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [16]:
import json

with open("data/ragas_eval_dataset.jsonl", "w", encoding="utf-8") as f:
    for sample in testset.to_list():
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

eval_df.to_csv("data/ragas_eval_dataset.csv", index=False, encoding="utf-8")

print("saved data/ragas_eval_dataset.csv and data/ragas_eval_dataset.jsonl")

saved data/ragas_eval_dataset.csv and data/ragas_eval_dataset.jsonl


In [17]:
eval_df.head(10)  # first 10 rows

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is NATIONAL FOOD SECURITY MISSION?,[## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL...,The context only mentions NATIONAL FOOD SECURI...,María Ortega,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,"How was the scheme implemented from 2007-08, a...",[The scheme is being implemented from 2007 ­ 0...,The scheme is being implemented from 2007-08 a...,Dr. Kavya Menon,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,Prices ka kya matlab hai aur ye kaise under ch...,[Objectives: Increase food security by steppin...,"Context ke mutabiq, objectives me ""prices rema...",Dr. Kavya Menon,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,Farm machineries distribution under NMSA and w...,[-  Cluster Demonstrations - test beds for de...,Farm Machineries refers to the distribution of...,Dr. Kavya Menon,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
4,What is the purpose of setting up project mana...,[-  Plant Protection - Subsidised distributio...,The context states that part of the program in...,María Ortega,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
5,How is the NMSA scheme implemented through the...,[## NATIONAL MISSION FOR SUSTAINABLE AGRICULTU...,The NMSA scheme is being implemented with diff...,Asha Verma,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
6,What centrally sponsored scheme mentioned in t...,[<1-hop>\n\n## CENTRALLY SPONSORED SCHEMES\n\n...,The centrally sponsored scheme mentioned in th...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,In the context of cluster demonstrations and s...,[<1-hop>\n\n-  Cluster Demonstrations - test ...,The context explains that cluster demonstratio...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,What does the context say about using ICT in a...,[<1-hop>\n\n## Interventions:\n\n-  Soil Heal...,The context presents ICT in agriculture and go...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,"Under NMSA, how does the PACS-linked process h...",[<1-hop>\n\n## 3.1.1 Compulsory Component\n\n-...,Compulsory coverage applies to all farmers who...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
